# Assessment 1: Mini RAG Pipeline with ChromaDB

In [1]:
!pip install -q chromadb sentence-transformers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.


## 1. Documents

In [2]:
docs = [
    "Python is a popular programming language used for data science and web development.",
    "The Eiffel Tower is a famous landmark located in Paris, France.",
    "Photosynthesis is the process by which plants convert sunlight into energy.",
    "Machine learning lets computers learn patterns from data without explicit programming.",
    "The Amazon rainforest is the largest tropical rainforest in the world.",
]

## 2. Preprocess (lowercase + remove punctuation)

In [3]:
import string

def preprocess(text):
    return text.lower().translate(str.maketrans("", "", string.punctuation))

clean_docs = [preprocess(d) for d in docs]
clean_docs

['python is a popular programming language used for data science and web development',
 'the eiffel tower is a famous landmark located in paris france',
 'photosynthesis is the process by which plants convert sunlight into energy',
 'machine learning lets computers learn patterns from data without explicit programming',
 'the amazon rainforest is the largest tropical rainforest in the world']

## 3. Embed documents

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(clean_docs).tolist()
print(len(embeddings), "embeddings of size", len(embeddings[0]))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

5 embeddings of size 384


## 4. Store in ChromaDB (cosine similarity)

In [5]:
import chromadb

client = chromadb.Client()
collection = client.create_collection(name="docs", metadata={"hnsw:space": "cosine"})

collection.add(
    ids=[f"doc{i}" for i in range(len(docs))],
    embeddings=embeddings,
    documents=docs,  # original text, returned as the response
)
print("Stored:", collection.count(), "documents")

Stored: 5 documents


## 5. Query, retrieve, respond

In [6]:
def answer(query):
    q_emb = model.encode([preprocess(query)]).tolist()
    res = collection.query(query_embeddings=q_emb, n_results=1)
    doc = res["documents"][0][0]
    score = 1 - res["distances"][0][0]  # cosine distance -> similarity
    return doc, score

query = input("Ask a question: ")
doc, score = answer(query)
print(f"\nResponse: {doc}\nCosine similarity: {score:.3f}")

Ask a question: Where is the Eiffel Tower?

Response: The Eiffel Tower is a famous landmark located in Paris, France.
Cosine similarity: 0.846
